# Landing Write Helper

Defines `write_to_landing()`, the single reusable write function used by every landing ingestion notebook.

- Purpose: Writes a source DataFrame into the landing lakehouse's Files section, tagging each row with its batch ID, ingestion timestamp, and source path for lineage
- Load types: `incremental` — writes into a batch-named subfolder, so re-running a batch only touches its own data; `full` — overwrites a fixed folder on every run
- Output format: Raw CSV, unmodified from source — no transformation happens at this layer
- Used by: `flight-landing`, `airport-landing`, `carrier-landing` (via `%run`)

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

def write_to_landing(
    input_df,
    target_folder,
    batch_id,
    source_path,
    load_type="incremental",
    file_format="csv"
):
    final_df = (input_df
        .withColumn("batch_id", F.lit(batch_id))
        .withColumn("ingested_timestamp", F.lit(datetime.now(timezone.utc)))
        .withColumn("source_path", F.lit(source_path))
    )

    if load_type == "incremental":
        output_path = f"{lakehouse_path}/{target_folder}/{batch_id}"
    else:
        output_path = f"{lakehouse_path}/{target_folder}"

    writer = final_df.write.format(file_format).mode("overwrite")
    if file_format == "csv":
        writer = writer.option("header", "true")

    writer.save(output_path)

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] [TooManyRequestsForCapacity] HTTP Response code 430: This Spark job can't be run because you've hit spark overall capacity compute limit. To proceed, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. For more visibility and control, go to Workspace settings → Job management (Job Concurrency & Queue Monitoring) to review running and queued Spark jobs, understand capacity contention, and take action as needed. [Learn more at 'https://go.microsoft.com/fwlink/?linkid=2356970&clcid=0x409']. HTTP status code: 430.